# **Day 39: Debugging and Logging**

**Debugging** means finding and fixing mistakes (bugs) in code. Every programmer spends a lot of time on it, so it is worth learning good habits.

## **1. Read the Error Message From the Bottom**
When Python shows an error (called a **traceback**):
- The **last line** tells **what** went wrong.
- The lines above show **where** it happened (which function called which).

For example, if the last line is `KeyError: 'C'`, it means we tried to use the key `'C'` that does not exist in a dictionary.

## **2. Print Values to Check Them**
The simplest way to debug is to print the values just before the problem. The `=` inside an f-string shows the **name and the value** together.

In [1]:
def buggy_average(nums):
    total = 0
    for i in range(1, len(nums)):     # Bug is here
        total += nums[i]
    print(f"{total = }, {len(nums) = }")
    return total / len(nums)

print(buggy_average([10, 20, 30]))    # We expected 20

total = 50, len(nums) = 3
16.666666666666668


The total is 50 instead of 60, so the loop **skipped the first number**. `range(1, ...)` should be `range(len(nums))`, or even simpler, use `sum(nums)`.

In [2]:
def fixed_average(nums):
    return sum(nums) / len(nums)

print(fixed_average([10, 20, 30]))

20.0


## **3. The Debugger: breakpoint()**
Putting `breakpoint()` in the code **pauses** the program at that line. Then we can look at the values step by step.

Useful commands in the debugger:
- `n`: go to the next line
- `p x`: print the value of `x`
- `c`: continue running
- `q`: quit

In Jupyter, we can also run `%debug` in a new cell just after an error happens.

In [3]:
# def find_max(nums):
#     best = nums[0]
#     for n in nums:
#         breakpoint()        # The program will pause here
#         if n > best:
#             best = n
#     return best
#
# find_max([3, 9, 2])

## **4. Logging: A Better Way Than print()**
**Logging** means writing messages about what the program is doing. It is better than `print()` because:
- Messages have **levels** (DEBUG, INFO, WARNING, ERROR, CRITICAL).
- We can **turn off** less important messages without deleting code.
- Messages can be saved to a **file** with the time.

| Level | Used for |
|---|---|
| DEBUG | Small details, only for the programmer |
| INFO | Normal progress ("file loaded") |
| WARNING | Something odd, but the program continues |
| ERROR | Something failed |
| CRITICAL | A very serious problem |

In [4]:
import logging, sys

logger = logging.getLogger("ndvi_app")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)-8s %(message)s"))
logger.addHandler(handler)

logger.debug("Reading band B04 and B08")
logger.info("2 bands loaded")
logger.warning("12% of pixels are cloudy")
logger.error("Band B11 file not found")
logger.critical("Disk is full!")

DEBUG    Reading band B04 and B08


INFO     2 bands loaded


WARNING  12% of pixels are cloudy


ERROR    Band B11 file not found


CRITICAL Disk is full!


In [5]:
# Change the level: now DEBUG and INFO messages are hidden
logger.setLevel(logging.WARNING)
logger.debug("You will not see this")
logger.info("Or this")
logger.warning("But you will see this")

WARNING  But you will see this


## **Saving Logs to a File**

In [6]:
file_logger = logging.getLogger("processing_log")
file_logger.setLevel(logging.INFO)
fh = logging.FileHandler("processing.log", mode="w", encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M"))
file_logger.addHandler(fh)

for scene, cloud in [("S2A_20240105", 12), ("S2B_20240110", 3), ("S2A_20240115", 48)]:
    if cloud > 20:
        file_logger.warning(f"{scene} skipped, cloud {cloud}%")
    else:
        file_logger.info(f"{scene} processed")
fh.close()

print(open("processing.log", encoding="utf-8").read())

2026-09-27 07:31 | INFO | S2A_20240105 processed
2026-09-27 07:31 | INFO | S2B_20240110 processed
2026-09-27 07:31 | WARNING | S2A_20240115 skipped, cloud 48%



A log file like this is very useful when a long job (like processing 500 satellite images overnight) runs without anyone watching. In the morning, we can read the log to see what happened.

## **A Simple Debugging Checklist**
1. Read the error message **fully**.
2. Try to repeat the error with the **smallest** input.
3. Check the **type** and value of your variables (`type()`, `print()`).
4. Explain your code line by line to a friend (or to a rubber duck!).
5. Take a short break. It helps more than you would think.

## **Practice Problem: Find the Bug**
This function should list the words that appear more than once.

In [7]:
def repeated_words(text):
    counts = {}
    for w in text.split():
        counts[w] = counts.get(w, 0) + 1
    return [w for w, c in counts.items() if c > 1]

print(repeated_words("Kolkata kolkata is kolkata"))

['kolkata']


It misses that "Kolkata" and "kolkata" are the same word. The fix is to make the text lower case first: `text.lower().split()`.

In [8]:
def repeated_words(text):
    counts = {}
    for w in text.lower().split():
        counts[w] = counts.get(w, 0) + 1
    return [w for w, c in counts.items() if c > 1]

print(repeated_words("Kolkata kolkata is kolkata"))

['kolkata']
